In [1]:
# R6_out_of_time.ipynb  (revision stage 6 - temporal robustness / out-of-time, addresses the
# reviewer's "random-split PSI approximately 0 is meaningless" limitation)
# Fourth dataset FAMILY: Polish companies bankruptcy (Zieba, Tomczak & Tomczak 2016),
# five cohorts 1year..5year differing in forecasting horizon and base rate (bankruptcy
# 3.9% at the 5-year-ahead horizon rising to 6.9% at the 1-year-ahead horizon), 64
# financial-ratio features with genuine missingness. This dataset is used NOT to
# re-derive the coefficient correspondence (established on GMSC/German/HELOC in R1/R1b;
# it is not identifiable here because 64 features x several bins approaches the number
# of bankruptcies, so the free multivariate dummy fit is near-separated) but to stress
# the two OPERATIONAL claims on a high-dimensional, extremely imbalanced, genuinely
# shifting family:
#   (A) WITHIN-HORIZON discrimination parity: logit-WoE vs stump-boosting AUC on each
#       cohort, in-fold binning (M6), equivalence by TOST (+-0.01).
#   (B) OUT-OF-TIME: fit the scorecard on one cohort, freeze WoE grid + coefficients,
#       deploy on another cohort. Out-of-time AUC for both models + PSI of the score
#       distribution between cohorts. Unlike a random split (PSI ~ 0), cross-cohort PSI
#       is materially > 0 - a real shift under which we verify (i) both models degrade
#       together (equivalence preserved) and (ii) the booster is not uniquely fragile.
# Robust (ridge-stabilised) logistic throughout. Single cell; greyscale; dpi600 png+pdf.

import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
warnings.filterwarnings('ignore')

SEED=42; IV_THRESH=0.02; N_SPLIT=10; HORIZONS=[1,2,3,4,5]; MARGIN=0.01
sns.set_theme(style='whitegrid', context='paper'); sns.set_palette('Greys')
plt.rcParams.update({'axes.edgecolor':'0.2','grid.color':'0.85','font.size':10})
ROOT=os.path.abspath(os.path.join(os.getcwd(),'..')); DATA=os.path.join(ROOT,'data')
TABLES=os.path.join(ROOT,'results','tables'); FIGS=os.path.join(ROOT,'results','figures')
def savefig(fig,name):
    for e in ('png','pdf'): fig.savefig(os.path.join(FIGS,name+'.'+e),dpi=600,bbox_inches='tight')
    plt.close(fig)

COH={h:pd.read_parquet(os.path.join(DATA,'polish_%dyear.parquet'%h)) for h in HORIZONS}
FEAT=[c for c in COH[1].columns if c.startswith('Attr')]
print('Polish cohorts (rows, bad%):',{h:(len(COH[h]),round(100*COH[h]['class'].mean(),2)) for h in HORIZONS})
print('features:',len(FEAT))

def woe_of(g,b,TG,TB): gr=(g+0.5)/(TG+0.5); br=(b+0.5)/(TB+0.5); return float(np.log(br/gr))
def tree_edges(x,yv,max_leaf=5,min_frac=0.05):
    m=x.notna(); xx=x[m].values.reshape(-1,1).astype(float); yy=yv[m.values]
    if len(np.unique(xx))<=2: return []
    t=DecisionTreeClassifier(max_leaf_nodes=max_leaf,min_samples_leaf=int(min_frac*len(xx)),random_state=SEED)
    t.fit(xx,yy); thr=t.tree_.threshold[t.tree_.feature>=0]; return sorted(float(v) for v in thr)
def build_maps(Xtr,ytr):
    TG=int((ytr==0).sum()); TB=int((ytr==1).sum()); min_n=max(int(0.02*len(ytr)),10); maps={}; iv={}
    for c in FEAT:
        def tbl(e):
            cuts=[-np.inf]+list(e)+[np.inf]; m=Xtr[c].notna(); yy=ytr[m.values]
            idx=pd.cut(Xtr[c][m],cuts,labels=False,include_lowest=True); rr=[]
            for b in range(len(cuts)-1):
                sel=idx==b; g=int((yy[sel]==0).sum()); bd=int((yy[sel]==1).sum())
                if g+bd>0: rr.append({'kind':'num','lo':cuts[b],'hi':cuts[b+1],'good':g,'bad':bd})
            return rr
        e=list(tree_edges(Xtr[c],ytr))
        while e:
            rr=tbl(e); br=np.array([r['bad']/(r['good']+r['bad']) for r in rr])
            if len(br)<=2 or np.all(np.diff(br)>=-1e-12) or np.all(np.diff(br)<=1e-12): break
            d=np.diff(br); s=np.sign(np.nansum(d)); e.pop(int(np.argmax(-s*d)))
        recs=tbl(e)
        seln=Xtr[c].isna(); gn=int((ytr[seln.values]==0).sum()); bn=int((ytr[seln.values]==1).sum())
        if gn+bn>0: recs.append({'kind':'nan','good':gn,'bad':bn})
        keep=[]; og=ob=0
        for r in recs:
            if r['kind']=='nan' and r['good']+r['bad']<min_n: og+=r['good']; ob+=r['bad']
            else: keep.append(r)
        if og+ob>0: keep.append({'kind':'other','good':og,'bad':ob})
        for r in keep: r['woe']=woe_of(r['good'],r['bad'],TG,TB); r['n']=r['good']+r['bad']
        maps[c]=keep
        iv[c]=float(np.sum([((r['bad']+.5)/(TB+.5)-(r['good']+.5)/(TG+.5))*r['woe'] for r in keep]))
    return maps,[c for c in FEAT if iv[c]>=IV_THRESH]
def to_idx(x,recs):
    idx=np.full(len(x),-1,int)
    for p,r in enumerate(recs):
        if r['kind']=='num':
            sel=x.notna().values&(x>r['lo']).values&(x<=r['hi']).values; idx[sel]=p
        elif r['kind']=='nan': idx[x.isna().values]=p
    oth=[p for p,r in enumerate(recs) if r['kind']=='other']; idx[idx==-1]=oth[0] if oth else 0
    return idx
def woe_design(Xdf,maps,keep):
    return np.column_stack([np.array([r['woe'] for r in maps[c]])[to_idx(Xdf[c],maps[c])] for c in keep])
# robust logistic on WoE grid (small ridge to absorb collinear WoE columns under separation)
def fit_logit(W,y): return LogisticRegression(C=1e4,max_iter=5000,solver='lbfgs').fit(W,y)
def fit_stump(W,y): return GradientBoostingClassifier(max_depth=1,learning_rate=0.3,n_estimators=300,random_state=SEED).fit(W,y)

# ===================== PART A: within-horizon discrimination parity =====================
auc_rows=[]; t0=time.time()
for h in HORIZONS:
    df=COH[h]; y=df['class'].values.astype(int)
    skf=StratifiedKFold(n_splits=N_SPLIT,shuffle=True,random_state=SEED)
    for fold,(tr,te) in enumerate(skf.split(df[FEAT],y)):
        Xtr=df.iloc[tr]; ytr=y[tr]; Xte=df.iloc[te]; yte=y[te]
        maps,keep=build_maps(Xtr,ytr)
        if len(keep)<2: continue
        Wtr=woe_design(Xtr,maps,keep); Wte=woe_design(Xte,maps,keep)
        ml=fit_logit(Wtr,ytr); ms=GradientBoostingClassifier(max_depth=1,learning_rate=0.3,
            n_estimators=300,random_state=SEED+fold).fit(Wtr,ytr)
        auc_rows.append([h,fold,roc_auc_score(yte,ml.predict_proba(Wte)[:,1]),
                         roc_auc_score(yte,ms.predict_proba(Wte)[:,1]),len(keep)])
    print('  h%d done (%.0fs)'%(h,time.time()-t0))
aucdf=pd.DataFrame(auc_rows,columns=['horizon','fold','auc_logit','auc_stump','n_feat'])
asum=aucdf.groupby('horizon').agg(logit=('auc_logit','mean'),logit_sd=('auc_logit','std'),
     stump=('auc_stump','mean'),stump_sd=('auc_stump','std'),n_feat=('n_feat','mean')).reset_index()
asum['diff']=asum['stump']-asum['logit']
d=aucdf['auc_stump'].values-aucdf['auc_logit'].values
se=d.std(ddof=1)/np.sqrt(len(d)); tl=(d.mean()+MARGIN)/se; tu=(d.mean()-MARGIN)/se
p_tost=max(stats.t.sf(tl,len(d)-1),stats.t.cdf(tu,len(d)-1))
print('\n=== PART A within-horizon discrimination parity ===')
print(asum.round(4).to_string(index=False))
print('pooled AUC diff(stump-logit)=%+.4f | TOST(+-0.01) p=%.2e'%(d.mean(),p_tost))
asum.round(6).to_csv(os.path.join(TABLES,'R6_polish_auc.csv'),index=False)
aucdf.round(6).to_csv(os.path.join(TABLES,'R6_polish_auc_long.csv'),index=False)

# ===================== PART B: out-of-time cross-cohort =====================
def psi(exp,act,bins=10):
    qs=np.quantile(exp,np.linspace(0,1,bins+1)); qs[0]=-np.inf; qs[-1]=np.inf
    e=np.histogram(exp,qs)[0]/len(exp); a=np.histogram(act,qs)[0]/len(act)
    e=np.clip(e,1e-4,None); a=np.clip(a,1e-4,None); return float(np.sum((a-e)*np.log(a/e)))
FIT={}
for h in HORIZONS:
    df=COH[h]; y=df['class'].values.astype(int); maps,keep=build_maps(df,y)
    Wtr=woe_design(df,maps,keep); ml=fit_logit(Wtr,y); ms=fit_stump(Wtr,y)
    FIT[h]=dict(maps=maps,keep=keep,ml=ml,ms=ms,s_l=ml.predict_proba(Wtr)[:,1],s_s=ms.predict_proba(Wtr)[:,1])
oot=[]
for ht in HORIZONS:
    f=FIT[ht]
    for he in HORIZONS:
        de=COH[he]; ye=de['class'].values.astype(int); Wde=woe_design(de,f['maps'],f['keep'])
        pl=f['ml'].predict_proba(Wde)[:,1]; ps=f['ms'].predict_proba(Wde)[:,1]
        oot.append([ht,he,roc_auc_score(ye,pl),roc_auc_score(ye,ps),psi(f['s_l'],pl),psi(f['s_s'],ps)])
ootdf=pd.DataFrame(oot,columns=['train_h','test_h','auc_logit','auc_stump','psi_logit','psi_stump'])
ootdf.round(6).to_csv(os.path.join(TABLES,'R6_oot_matrix.csv'),index=False)
off=ootdf[ootdf.train_h!=ootdf.test_h]; diag=ootdf[ootdf.train_h==ootdf.test_h]
print('\n=== PART B out-of-time (cross-cohort) ===')
print('in-sample (diag) AUC : logit %.3f  stump %.3f'%(diag.auc_logit.mean(),diag.auc_stump.mean()))
print('out-of-time AUC      : logit %.3f  stump %.3f  (drop %.3f / %.3f)'
      %(off.auc_logit.mean(),off.auc_stump.mean(),
        diag.auc_logit.mean()-off.auc_logit.mean(),diag.auc_stump.mean()-off.auc_stump.mean()))
print('cross-cohort PSI     : logit %.3f  stump %.3f  (random-split PSI ~ 0)'
      %(off.psi_logit.mean(),off.psi_stump.mean()))
dd=off.auc_stump.values-off.auc_logit.values
se=dd.std(ddof=1)/np.sqrt(len(dd)); tl=(dd.mean()+MARGIN)/se; tu=(dd.mean()-MARGIN)/se
p_oot=max(stats.t.sf(tl,len(dd)-1),stats.t.cdf(tu,len(dd)-1))
print('out-of-time AUC diff(stump-logit)=%+.4f  TOST(+-0.01) p=%.2e'%(dd.mean(),p_oot))

# ===================== FIGURES =====================
# (1) within-horizon AUC by cohort (logit vs stump)
fig,ax=plt.subplots(figsize=(7.2,4.4))
xp=np.arange(len(HORIZONS))
ax.bar(xp-0.2,asum['logit'],width=0.4,yerr=asum['logit_sd'],capsize=3,color='0.3',label='logistic-WoE')
ax.bar(xp+0.2,asum['stump'],width=0.4,yerr=asum['stump_sd'],capsize=3,color='0.65',label='stump boosting')
ax.set_xticks(xp); ax.set_xticklabels(['%d-year'%h for h in HORIZONS])
ax.set_ylim(0.75,0.97); ax.set_ylabel('AUC (10-fold, in-fold binning)')
ax.set_xlabel('forecasting-horizon cohort')
ax.legend(frameon=True,facecolor='white',edgecolor='0.5',fontsize=9,loc='upper left')
savefig(fig,'R6_polish_auc')
# (2) out-of-time: left AUC parity scatter; right PSI bars
fig,axes=plt.subplots(1,2,figsize=(11,4.6))
ax=axes[0]
lo,hi=0.60,0.95; ax.plot([lo,hi],[lo,hi],color='0.0',lw=1,ls='--')
ax.scatter(off.auc_logit,off.auc_stump,s=42,marker='o',facecolor='0.62',edgecolor='0.0',label='out-of-time pair')
ax.scatter(diag.auc_logit,diag.auc_stump,s=70,marker='*',facecolor='0.0',edgecolor='0.0',label='in-sample')
ax.set_xlim(lo,hi); ax.set_ylim(lo,hi)
ax.set_xlabel('logistic-WoE AUC'); ax.set_ylabel('stump-boosting AUC')
ax.set_title('cross-cohort deployment: AUC parity',fontsize=10)
ax.legend(frameon=True,facecolor='white',edgecolor='0.5',fontsize=8,loc='upper left')
ax=axes[1]
lab=['%d→%d'%(r.train_h,r.test_h) for _,r in off.iterrows()]; xpos=np.arange(len(off))
ax.bar(xpos-0.2,off.psi_logit,width=0.4,color='0.3',label='logistic-WoE')
ax.bar(xpos+0.2,off.psi_stump,width=0.4,color='0.65',label='stump boosting')
ax.axhline(0.1,color='0.0',lw=0.8,ls=':'); ax.axhline(0.25,color='0.0',lw=0.8,ls='--')
ax.set_xticks(xpos); ax.set_xticklabels(lab,rotation=90,fontsize=6)
ax.set_ylabel('PSI (train cohort $\\to$ deploy cohort)'); ax.set_title('cross-cohort score shift (PSI)',fontsize=10)
ax.legend(frameon=True,facecolor='white',edgecolor='0.5',fontsize=8)
savefig(fig,'R6_out_of_time')
print('\nfigures: R6_polish_auc, R6_out_of_time | tables: R6_polish_auc(.long), R6_oot_matrix')
print('R6 complete.')


Polish cohorts (rows, bad%): {1: (7027, np.float64(3.86)), 2: (10173, np.float64(3.93)), 3: (10503, np.float64(4.71)), 4: (9792, np.float64(5.26)), 5: (5910, np.float64(6.94))}
features: 64


  h1 done (21s)


  h2 done (54s)


  h3 done (89s)


  h4 done (133s)


  h5 done (151s)

=== PART A within-horizon discrimination parity ===
 horizon  logit  logit_sd  stump  stump_sd  n_feat   diff
       1 0.8923    0.0235 0.8972    0.0274    61.5 0.0049
       2 0.8273    0.0365 0.8322    0.0388    59.6 0.0049
       3 0.8610    0.0229 0.8670    0.0228    60.3 0.0060
       4 0.8793    0.0255 0.8890    0.0231    63.9 0.0097
       5 0.9172    0.0303 0.9281    0.0233    64.0 0.0110
pooled AUC diff(stump-logit)=+0.0073 | TOST(+-0.01) p=7.65e-02



=== PART B out-of-time (cross-cohort) ===
in-sample (diag) AUC : logit 0.894  stump 0.904
out-of-time AUC      : logit 0.830  stump 0.838  (drop 0.064 / 0.066)
cross-cohort PSI     : logit 0.092  stump 0.086  (random-split PSI ~ 0)
out-of-time AUC diff(stump-logit)=+0.0082  TOST(+-0.01) p=2.27e-01



figures: R6_polish_auc, R6_out_of_time | tables: R6_polish_auc(.long), R6_oot_matrix
R6 complete.
